# Genome PRD v1 — 05 Artifact and manifest

Verify the trained model and results, derive the manifest from canonical configuration and saved evidence, write the PRD pointer, and validate it. No manifest fields are hand-written.

In [1]:
from pathlib import Path
import json
import sys
from xgboost import XGBClassifier

here = Path.cwd().resolve()
ROOT = next(path for path in (here, *here.parents) if (path / 'src/training/prd_config.py').is_file())
sys.path.insert(0, str(ROOT))
from src.training import prd_config
from src.training.build_prd_manifest import build_prd_manifest, write_prd_manifest
from src.training.modeling import sha256_file
from src.training.prd import validate_prd_manifest

## Verify saved artifacts and feature contract

In [2]:
assert prd_config.PRD_ARTIFACT_PATH.is_file()
assert prd_config.PRD_RESULTS_PATH.is_file()
model = XGBClassifier()
model.load_model(prd_config.PRD_ARTIFACT_PATH)
results = json.loads(prd_config.PRD_RESULTS_PATH.read_text())
assert model.get_booster().num_features() == len(prd_config.PRD_FEATURES) == 25
assert results['feature_columns'] == list(prd_config.PRD_FEATURES)
assert results['model_params'] == prd_config.serialized_model_params()
print('Model SHA-256:', sha256_file(prd_config.PRD_ARTIFACT_PATH))

Model SHA-256: 53fdfbd3b8b3104dc4cbb74598a40b5f3a206bd09ad3d6d7093481046b4a989a


## Build, write, and validate the canonical PRD manifest

In [3]:
manifest = build_prd_manifest(
    model_path=prd_config.PRD_ARTIFACT_PATH,
    results_path=prd_config.PRD_RESULTS_PATH,
    comparison_path=prd_config.PRD_COMPARISON_PATH,
    experiment_candidate_results_path=prd_config.PRD_CANDIDATE_RESULTS_PATH,
    baseline_results_path=prd_config.PRD_BASELINE_RESULTS_PATH,
    v2_validation_path=prd_config.PRD_V2_VALIDATION_PATH,
    run_manifest_path=prd_config.PRD_RUN_MANIFEST_PATH,
    promotion_date=prd_config.PRD_PROMOTION_DATE,
    validation_scope=prd_config.PRD_VALIDATION_SCOPE,
    promote=True,
)
write_prd_manifest(manifest, prd_config.PRD_MODEL_MANIFEST_PATH)
validated = validate_prd_manifest(manifest)
display({'artifact': validated['artifact']['path'], 'results': validated['results_artifact']['path'], 'manifest': str(prd_config.PRD_MODEL_MANIFEST_PATH.relative_to(ROOT)), 'status': validated['status']})

{'artifact': 'models/prd/xgboost_genome_prd_v1.model.json',
 'results': 'models/prd/xgboost_genome_prd_v1.results.json',
 'manifest': 'models/prd/prd_model_manifest.json',
 'status': 'PRD'}